In [1]:
pip install datasets

In [2]:
from huggingface_hub import login

In [ ]:
login("")

In [21]:
from datasets import load_dataset, Dataset

dataset = load_dataset(
    "laion/laion400m",
    split="train",
    streaming=True
)

dataset = dataset.shuffle(buffer_size=10000, seed=42)

sample = list(dataset.take(100000))

Resolving data files:   0%|          | 0/128 [00:00<?, ?it/s]

In [22]:
objects = [
    "polar bear", "chalkboard", "banana",
    "apple", "dog", "chair", "flower", "shoe",
    "bag", "frog", "ball", "car"
]

colors = [
    "blue", "yellow", "red",
    "green", "pink", "orange",
    "purple", "white", "black"
]

In [23]:
!pip install seaborn pandas matplotlib -q

In [24]:
from collections import defaultdict
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
def contains_word(text, word):
    """
    Verifica se `word` ocorre como palavra isolada em `text`.
    re.IGNORECASE garante robustez mesmo que o loop já faça .lower(),
    pois contains_word pode ser chamado antes da normalização.
    """
    return re.search(rf"\b{word}\b", text, re.IGNORECASE) is not None


def object_color_pair(text, obj, color):
    """
    Detecta binding real entre objeto e cor usando três padrões sintáticos.

    Padrões:
        p1 — adjetivo direto pré-nominal : "blue car", "pink chalkboard"
        p2 — verbo copulativo            : "the car is/was/turned red"
             (lista expandida: is, are, was, were, looks, appears, seems,
              turned, became, got, remained, stays + variantes)
        p4 — possessivo                  : "the frog's green skin"
        p5 — hifen + adjetivo derivado   : "blue-colored car", "red-painted wall"

    Remoção do pattern3 original:
        O padrão anterior `obj(?:\s+\w+){0,2}\s+color` era excessivamente
        permissivo e gerava falsos positivos em construções como:
          - "dog and black cat"    (obj + conjunção + cor de outro referente)
          - "flower with pink petals"  (cor pertence ao substantivo seguinte)
          - "apple was not green"  (negação capturada como binding)
        A remoção reduz recall em casos ambíguos, mas aumenta
        significativamente a precisão — mais adequado para estimar viés
        de dataset onde falsos positivos distorcem as marginais do PPMI.
    """
    # p1: adjetivo direto — "green frog", "blue polar bear"
    p1 = rf"\b{color}\s+{obj}\b"

    # p2: verbo copulativo — lista expandida (was/were/turned/became cobrem
    #     construções de tempo passado ausentes na versão original)
    verbs = [
        "is", "are", "was", "were",
        "looks?", "appears?", "seems?",
        "turned?", "became?", "got", "remained?", "stays?",
    ]
    verb_pattern = "|".join(verbs)
    p2 = rf"\b{obj}\b\s+(?:{verb_pattern})\s+{color}\b"

    # p4: possessivo — "the frog's green skin"
    p4 = rf"\b{obj}'s\s+{color}\b"

    # p5: hifen + adjetivo derivado — "blue-colored car", "red-painted wall"
    p5 = rf"\b{color}[-\s]+(?:colored?|painted?|hued?)\s+{obj}\b"

    return any(
        re.search(p, text, re.IGNORECASE)
        for p in [p1, p2, p4, p5]
    )


In [ ]:
counts_base = defaultdict(lambda: defaultdict(int))  # co-ocorrência simples (obj + cor na mesma caption)
object_counts_base = defaultdict(int)                # quantas captions contêm cada objeto

counts_pattern = defaultdict(lambda: defaultdict(int)) # binding real (padrão sintático)
object_counts_pattern = defaultdict(int)

# contagem global de cores no corpus completo (não apenas junto a objetos)
# Necessário para calcular P(color) corretamente na fórmula do PPMI
color_counts_global = defaultdict(int)
N_valid = 0  # total de captions válidas


In [ ]:
import math
import numpy as np

N = 100000
teste = []
index = 0

for sample in dataset.take(N):
    caption = sample["caption"]
    if caption is None:
        continue

    N_valid += 1

    if contains_word(caption, "banana") and contains_word(caption, "pink"):
        teste.append(caption)

    caption_lower = caption.lower()

    # ── CONTAGEM GLOBAL DE CORES ──
    for color in colors:
        if contains_word(caption_lower, color):
            color_counts_global[color] += 1

    for obj in objects:
        if contains_word(caption_lower, obj):

            # ===== BASELINE (co-ocorrência simples) =====
            object_counts_base[obj] += 1
            for color in colors:
                if contains_word(caption_lower, color):
                    counts_base[obj][color] += 1

            # ===== PATTERN (binding real) =====
            object_counts_pattern[obj] += 1
            for color in colors:
                if object_color_pair(caption_lower, obj, color):
                    counts_pattern[obj][color] += 1

    index += 1

for t in teste:
    print(f"Teste: {t}")

print("\n=== DEBUG ===")
print("N_valid:", N_valid)
print("object_counts_pattern:", dict(object_counts_pattern))

# =========================
# PROBABILIDADES CONDICIONAIS  P(color | obj)
# =========================

def compute_probabilities(counts, object_counts):
    probs = {}
    for obj in objects:
        total = object_counts[obj]
        if total == 0:
            continue
        probs[obj] = {color: counts[obj][color] / total for color in colors}
    return probs

probs_base    = compute_probabilities(counts_base,    object_counts_base)
probs_pattern = compute_probabilities(counts_pattern, object_counts_pattern)

# =========================
# DATAFRAMES
# =========================

df_base    = pd.DataFrame(probs_base).fillna(0)
df_pattern = pd.DataFrame(probs_pattern).fillna(0)

# =========================
# HEATMAPS BASELINE vs PATTERN
# =========================

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.heatmap(df_base, annot=True, fmt=".2f", cmap="viridis")
plt.title("Baseline (co-ocorrência)")

plt.subplot(1, 2, 2)
sns.heatmap(df_pattern, annot=True, fmt=".2f", cmap="viridis")
plt.title("Pattern (binding real)")

plt.tight_layout()
plt.show()

# =========================
# BINDING STRENGTH & GAP
# =========================

binding_strength = {}
binding_gap      = {}

for obj in objects:
    binding_strength[obj] = {}
    binding_gap[obj]      = {}
    for color in colors:
        b = probs_base.get(obj, {}).get(color, 0)
        p = probs_pattern.get(obj, {}).get(color, 0)
        binding_strength[obj][color] = (p / b) if b > 0 else 0
        binding_gap[obj][color] = b - p

# =========================
# MARGINAIS CORRIGIDAS PARA PPMI
# =========================

P_obj = {
    obj: object_counts_pattern[obj] / N_valid
    for obj in objects
}

P_color = {
    color: color_counts_global[color] / N_valid
    for color in colors
}

P_joint = {
    obj: {
        color: counts_pattern[obj][color] / N_valid
        for color in colors
    }
    for obj in objects
}

# =========================
# PPMI, NPMI e PMI_RAW
# =========================

binding_ppmi  = {}
binding_npmi  = {}
binding_pmi   = {}   # PMI sem truncagem — usado no heatmap de viés negativo
binding_score = {}

# ── PASSAGEM 1: PMI/NPMI apenas para pares com co-ocorrência > 0 ──
# Pares com p_joint == 0 ficam pendentes; preenchidos na passagem 2
# com sentinela coerente (PMI = piso negativo real, NPMI = -1).

pending_zero = []  # (obj, color) com p_joint == 0 mas marginais válidas

for obj in objects:
    binding_ppmi[obj]  = {}
    binding_npmi[obj]  = {}
    binding_pmi[obj]   = {}
    binding_score[obj] = {}

    for color in colors:
        p_joint = P_joint[obj][color]
        p_o     = P_obj.get(obj, 0)
        p_c     = P_color.get(color, 0)

        if p_o == 0 or p_c == 0:
            # objeto ou cor ausente do corpus -> indefinido, neutro
            binding_pmi[obj][color]   = 0.0
            binding_ppmi[obj][color]  = 0.0
            binding_npmi[obj][color]  = 0.0
            binding_score[obj][color] = 0.0
        elif p_joint == 0:
            # par válido que NUNCA co-ocorre: PMI -> -inf
            pending_zero.append((obj, color))
            binding_ppmi[obj][color]  = 0.0
            binding_npmi[obj][color]  = -1.0   # por definição
            binding_score[obj][color] = 0.0
            # binding_pmi preenchido na passagem 2
        else:
            pmi  = math.log(p_joint / (p_o * p_c))
            ppmi = max(0.0, pmi)
            npmi = pmi / (-math.log(p_joint))
            binding_pmi[obj][color]   = pmi
            binding_ppmi[obj][color]  = ppmi
            binding_npmi[obj][color]  = npmi
            binding_score[obj][color] = p_joint * ppmi

# ── PASSAGEM 2: piso de PMI para pares de co-ocorrência zero ──
# Piso = menor PMI observado - 1.0, garantindo que "nunca co-ocorre"
# seja sempre mais negativo que qualquer par só sub-representado.

observed_pmi = [
    binding_pmi[o][c]
    for o in objects for c in colors
    if c in binding_pmi[o]
]
min_observed_pmi = min(observed_pmi) if observed_pmi else -1.0
pmi_floor = min_observed_pmi - 1.0

for obj, color in pending_zero:
    binding_pmi[obj][color] = pmi_floor

print(f"\n[PMI] menor PMI observado = {min_observed_pmi:.3f} | "
      f"piso para pares de co-ocorrência zero = {pmi_floor:.3f}")

# =========================
# HEATMAP PPMI
# =========================

df_ppmi = pd.DataFrame(binding_ppmi).fillna(0)
plt.figure(figsize=(10, 7))
sns.heatmap(df_ppmi, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("PPMI (associação semântica) — corrigido")
plt.tight_layout()
plt.show()

# =========================
# HEATMAP NPMI COMPLETO
# =========================

df_npmi = pd.DataFrame(binding_npmi).fillna(0)
plt.figure(figsize=(10, 7))
sns.heatmap(df_npmi, annot=True, fmt=".2f", cmap="RdBu", center=0, vmin=-1, vmax=1)
plt.title("NPMI (normalizado, [-1, 1])\nAzul = associação positiva | Vermelho = sub-representação")
plt.tight_layout()
plt.show()

# =========================
# HEATMAP BINDING SCORE — escala log para legibilidade
# =========================

df_score     = pd.DataFrame(binding_score).fillna(0)
df_score_log = df_score.apply(lambda col: col.map(lambda v: math.log1p(v * 1e4)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Esquerda: valores brutos (escala original, referência)
sns.heatmap(df_score, annot=True, fmt=".2e", cmap="magma", ax=axes[0])
axes[0].set_title("Binding Score — escala linear\n(P_joint × PPMI)")

# Direita: log1p para destacar diferenças entre pares com score próximo de zero
sns.heatmap(df_score_log, annot=True, fmt=".3f", cmap="magma", ax=axes[1])
axes[1].set_title("Binding Score — escala log1p(score × 1e4)\n(destaca diferenças entre valores baixos)")

plt.tight_layout()
plt.show()

# =========================
# HEATMAP VIÉS NEGATIVO (foco em sub-representação)
# Mostra apenas pares onde PMI < 0 — evidência direta do viés do dataset
# =========================

# Usa NPMI (escala uniforme [-1,1]) em vez do PMI bruto: o NPMI já
# distingue corretamente "nunca co-ocorre" (= -1) de "apenas
# sub-representado" (valores intermediários), e não depende do piso
# artificial aplicado ao PMI dos pares de contagem zero.

df_npmi_bias = pd.DataFrame(binding_npmi).fillna(0)

# Mantém só o viés negativo: zera (branco) pares com NPMI >= 0
df_bias = df_npmi_bias.copy()
df_bias[df_bias >= 0] = 0

plt.figure(figsize=(11, 7))
sns.heatmap(
    df_bias,
    annot=True, fmt=".2f",
    cmap="Reds_r",   # vermelho mais intenso = mais sub-representado
    vmin=-1, vmax=0,
    linewidths=0.3
)
plt.title(
    "Viés Negativo do Dataset — NPMI < 0\n"
    "Vermelho escuro (-1) = par NUNCA co-ocorre (viés extremo)\n"
    "Vermelho claro = sub-representado | Branco = associação neutra/positiva"
)
plt.tight_layout()
plt.show()

# ── Rankings separados: viés extremo vs. sub-representação parcial ──
# Antes, ambos eram misturados num único ranking ordenado por NPMI,
# o que enchia o topo só com empates em -1.0. Agora separamos:
#   (a) pares que NUNCA co-ocorrem  (NPMI == -1)
#   (b) pares sub-representados mas existentes  (-1 < NPMI < 0)

never  = []
under  = []
for obj in objects:
    for color in colors:
        npmi_val = binding_npmi[obj][color]
        if npmi_val == -1.0:
            never.append((color, obj))
        elif npmi_val < 0:
            under.append((npmi_val, color, obj))

print(f"\n=== (a) PARES QUE NUNCA CO-OCORREM (NPMI = -1) : {len(never)} pares ===")
for color, obj in never[:20]:
    print(f"  {color:<8} + {obj}")
if len(never) > 20:
    print(f"  ... (+{len(never) - 20} outros)")

print(f"\n=== (b) TOP 15 SUB-REPRESENTADOS MAS EXISTENTES (-1 < NPMI < 0) ===")
under.sort()  # mais negativo primeiro
for npmi_v, color, obj in under[:15]:
    print(f"  {color:<8} + {obj:<12} | npmi={npmi_v:+.3f}")

# =========================
# GLOBAL BINDING SCORE
# =========================

all_scores     = [binding_score[obj][color] for obj in objects for color in colors]
nonzero_scores = [s for s in all_scores if s > 0]

global_mean         = sum(all_scores) / len(all_scores)
global_mean_nonzero = sum(nonzero_scores) / len(nonzero_scores) if nonzero_scores else 0
coverage            = len(nonzero_scores) / len(all_scores)

# Entropia do NPMI: mede o quão desigual é a distribuição de associações
npmi_vals    = [binding_npmi[obj][color] for obj in objects for color in colors]
npmi_pos     = [v for v in npmi_vals if v > 0]
npmi_entropy = -sum((v / sum(npmi_pos)) * math.log(v / sum(npmi_pos))
                    for v in npmi_pos if v > 0) if npmi_pos else 0

print(f"\nGlobal Binding Score (média total, inclui zeros):  {global_mean:.6f}")
print(f"Global Binding Score (média pares não-zero):        {global_mean_nonzero:.6f}")
print(f"Cobertura (fração de pares com binding > 0):        {coverage:.3f} ({len(nonzero_scores)}/{len(all_scores)} pares)")
print(f"Entropia dos NPMI positivos (quão concentrado):     {npmi_entropy:.3f}  (0=uniforme, alto=concentrado em poucos pares)")

# =========================
# DETALHES POR OBJETO
# =========================

for obj in objects:
    if obj in probs_pattern:
        print(f"\n=== {obj} ===")
        for color in colors:
            b      = probs_base.get(obj, {}).get(color, 0)
            p      = probs_pattern.get(obj, {}).get(color, 0)
            s      = binding_strength[obj][color]
            gap    = binding_gap[obj][color]
            ppmi   = binding_ppmi[obj][color]
            npmi   = binding_npmi[obj][color]
            score  = binding_score[obj][color]
            print(
                f"  {color:<8}: b={b:.3f} | p={p:.3f} | s={s:.3f} | gap={gap:.3f} "
                f"| ppmi={ppmi:.3f} | npmi={npmi:.3f} | score={score:.5f}"
            )

# =========================
# PREDIÇÃO DE ERRO (PPMI + NPMI)
# =========================

print("\n=== PREDIÇÃO DE ERRO ===")

test_cases = [
    ("banana",     "blue"),
    ("banana",     "yellow"),
    ("frog",       "green"),
    ("apple",      "white"),
    ("chalkboard", "pink"),
    ("car",        "red"),
]

for obj, color in test_cases:
    score = binding_score.get(obj, {}).get(color, 0)
    npmi  = binding_npmi.get(obj,  {}).get(color, 0)
    pmi   = binding_pmi.get(obj,   {}).get(color, 0)

    print(f"\nPrompt: '{color} {obj}'")
    print(f"  binding_score={score:.5f} | pmi={pmi:+.3f} | npmi={npmi:+.3f}")

    if npmi > 0.1:
        print("  → Modelo deve ACERTAR (associação forte no dataset)")
    elif npmi > 0.0:
        print("  → Modelo pode ser inconsistente (associação fraca)")
    elif npmi == -1.0:
        print("  → Modelo deve ERRAR: par NUNCA ocorre no dataset (viés extremo)")
    else:
        print("  → Modelo deve ERRAR ou ignorar atributo (sub-representado no dataset)")

# =========================
# EXEMPLOS DE CAPTIONS
# =========================

examples = []
for sample in dataset:
    caption = sample["caption"]
    if caption is None:
        continue
    caption_lower = caption.lower()
    if "green frog" in caption_lower:
        examples.append(caption_lower)
    if len(examples) == 20:
        break

print("\nExemplos de 'green frog':")
for e in examples:
    print(" -", e)
